In [1]:
import os
import sys
import glob
import multiprocessing
import subprocess
import time
import typing
from datetime import datetime as dt
from os import path
from IPython.display import clear_output

In [2]:
#input parameters

problems_name_dir: str = "../problems/benchmark/**/*.gz"
submat_size_lim: int = 512 #maximal size of square submat to generate conic cover cuts
row_trials_num: int = 5 #number of row trials to generate cuts
cuts_gen_rounds: int = 1 #number of cuts generation iterations
available_cpu_cores: int = 30
lazy_cuts_option: str = "off" #"on", "off" or "hybrid"
scaling: str = "off"
model_cuts_switch: str = "off" #switch for model cuts option
det_limit: int = 1024 #limit on maximum determinant to stard enumerative cuts generation

writing_cuts_on: bool = False
pureint_cuts_generation_on: bool = False
Hermite_cuts_generation_on: bool = False

In [3]:
#internal parameters and constants

cplex_workers: int = available_cpu_cores
cuts_gen_tl: int = 0 #cuts gen time limit in minutes
cplex_tl: int = 180 #cplex time limit in minutes
node_lim: int = -1 #node limit for CPlex
min_dim: int = 0 #minimal problem dim
max_dim: int = 0 #maximal problem dim

log_file_name_suff: str = "_log.txt" #out log file name suffix
cuts_file_name_suff: str = "_cuts.txt" #out cuts file name

out_dir_pref: str = "cutsPower__"
run_path: str = "../build/" 

cuts_gen_sec_tl: int = cuts_gen_tl * 60 #cuts gen time limit in seconds
cplex_sec_tl: int = cplex_tl * 60 #cplex time limit in seconds

In [4]:
# cut generation options

cut_generation_options: [str] = ["mir", "split"]

if pureint_cuts_generation_on : 
    cut_generation_options.append("pureint")

if Hermite_cuts_generation_on : 
    cut_generation_options.append("hermite")

split_cut_generation_options: [str] = ["gready", "fract"]

In [5]:
# creating output directory

def form_out_dir_name():
    time_now: str = dt.now().strftime("%Y.%m.%d_%H-%M-%S")
    return out_dir_pref + time_now

out_dir_name: str = form_out_dir_name()
print(f"Output directory name: {out_dir_name}")

os.mkdir(run_path+out_dir_name)
print(f"The outpud directory ({run_path+out_dir_name}) is created!")

Output directory name: cutsPower__2023.11.23_17-41-40
The outpud directory (../build/cutsPower__2023.11.23_17-41-40) is created!


In [6]:
# functions to generate run-string
def generate_run_str(file_name: str, rel_path: str, additional_run_options: typing.Dict[str,str]) -> str:

    file_options_str: str = str(list(additional_run_options.values()))
    add_run_options_str: str = ""

    for option_name, option_value in additional_run_options.items():
        add_run_options_str += f" {option_name} {option_value}" 

    log_rel_path: str = "./" + out_dir_name + "/" + file_name + "_" + file_options_str + "_" + log_file_name_suff 

    out_cuts_rel_path: str = "no"
    if writing_cuts_on : 
        out_cuts_rel_path = "./" + out_dir_name + "/" + file_name + "_" + file_options_str + "_" + cuts_file_name_suff

    run_str: str = f"./CPlexConsole -i {rel_path} -o {log_rel_path} --cuts_output {out_cuts_rel_path} --wcgn 1 --wcpl {cplex_workers} " \
        f"--iters {cuts_gen_rounds} --node_lim {node_lim} --model_cuts {model_cuts_switch} --trials_num {row_trials_num} --submat_limit {submat_size_lim} " \
        f"--cplex_tl {cplex_sec_tl} --cuts_tl {cuts_gen_sec_tl} --min_dim {min_dim} --max_dim {max_dim} --det_lim {det_limit}"
    
    run_str += add_run_options_str

    return run_str

def generate_baseline_run_str(file_name: str, rel_path: str) -> str:
    log_rel_path: str = "./" + out_dir_name + "/" + file_name + "_baseline_" + log_file_name_suff

    run_str: str = f"./CPlexConsole -b -i {rel_path} -o {log_rel_path} --wcgn 1 --wcpl {cplex_workers} " \
    f"--cplex_tl {cplex_sec_tl} --min_dim {min_dim} --max_dim {max_dim}"

    return run_str

In [7]:
# creating list of tasks to run
tasks_to_run : [str] = []

print("Problems to work with:")

for rel_path in glob.iglob(problems_name_dir, recursive=True):
    print(rel_path)
    file_name: str = path.basename(rel_path)

    tasks_to_run.append(generate_baseline_run_str(file_name, rel_path))

    for cut_generation_option in cut_generation_options:
        add_options_dict: typing.Dict[str, str] = {}
        add_options_dict["--cuts_fun"] = cut_generation_option

        if cut_generation_option == "split" :
            for split_cut_option in split_cut_generation_options:
                add_options_dict["--split_mode"] = split_cut_option
                tasks_to_run.append(generate_run_str(file_name, rel_path, add_options_dict ))
        else:
            tasks_to_run.append(generate_run_str(file_name, rel_path, add_options_dict))

Problems to work with:
../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz
../problems/benchmark/miplib2010_decomp2/instances/decomp2.mps.gz
../problems/benchmark/miplib2010_netdiversion/instances/netdiversion.mps.gz
../problems/benchmark/Andrea_Arias/instances/cmflsp50-24-8-8.mps.gz
../problems/benchmark/Michael_Bastubbe2/instances/graph20-20-1rand.mps.gz
../problems/benchmark/miplib2010_Robert_Fourer/instances/assign1-5-8.mps.gz
../problems/benchmark/miplib2010_sing/instances/sing326.mps.gz
../problems/benchmark/miplib2010_sing/instances/sing44.mps.gz
../problems/benchmark/miplib2010_50v-10/instances/50v-10.mps.gz
../problems/benchmark/miplib2010_roc/instances/rocII-5-11.mps.gz
../problems/benchmark/miplib2010_roc/instances/rocI-4-11.mps.gz
../problems/benchmark/miplib2010_lectsched-5-obj/instances/lectsched-5-obj.mps.gz
../problems/benchmark/miplib2010_csched008/instances/csched008.mps.gz
../problems/benchmark/miplib2010_rococoC10-001000/instances/rococoC10-001000.mps.

In [8]:
# printing list of tasks to run

print(f"There are {len(tasks_to_run)} tasks to run:\n")
for task_str in tasks_to_run:
    print(f"{task_str}\n")

There are 960 tasks to run:

./CPlexConsole -b -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_baseline__log.txt --wcgn 1 --wcpl 30 --cplex_tl 10800 --min_dim 0 --max_dim 0

./CPlexConsole -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_['mir']__log.txt --cuts_output no --wcgn 1 --wcpl 30 --iters 1 --node_lim -1 --model_cuts off --trials_num 5 --submat_limit 512 --cplex_tl 10800 --cuts_tl 0 --min_dim 0 --max_dim 0 --det_lim 1024 --cuts_fun mir

./CPlexConsole -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_['split', 'gready']__log.txt --cuts_output no --wcgn 1 --wcpl 30 --iters 1 --node_lim -1 --model_cuts off --trials_num 5 --submat_limit 512 --cplex_tl 10800 --cuts_tl 0 --min_dim 0 --max_dim 0 --det_lim 1024 --cuts_fun split --split_mode gready

./CPlexConsole -i ../probl

In [9]:
#runing the tasks

def run_task(run_str : str):
    return subprocess.run(run_str, shell=True, cwd=run_path)

starting_time = dt.now()
done_tasks_counter : int = 0

for task in tasks_to_run:
    task_result = run_task(task)

    args: str = str(task_result.args)
    if task_result.returncode == 0:
        print(f"({done_tasks_counter}) The process is successfully finished!, args are:")
    else:
        print(f"({done_tasks_counter}) The process is failed!, return code is {task_result.returncode}, args are:")
    print(f"{args}\n")

    done_tasks_counter += 1
    if done_tasks_counter % 10 == 0:
        clear_output(wait=True)
        print(f"-------{done_tasks_counter} tasks are done, {len(tasks_to_run)-done_tasks_counter} in rest-------\n")

(0) The process is successfully finished!, args are:
./CPlexConsole -b -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_baseline__log.txt --wcgn 1 --wcpl 30 --cplex_tl 10800 --min_dim 0 --max_dim 0

(1) The process is successfully finished!, args are:
./CPlexConsole -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_['mir']__log.txt --cuts_output no --wcgn 1 --wcpl 30 --iters 1 --node_lim -1 --model_cuts off --trials_num 5 --submat_limit 512 --cplex_tl 10800 --cuts_tl 0 --min_dim 0 --max_dim 0 --det_lim 1024 --cuts_fun mir

(2) The process is successfully finished!, args are:
./CPlexConsole -i ../problems/benchmark/miplib2010_timtab1/instances/timtab1.mps.gz -o ./cutsPower__2023.11.23_17-41-40/timtab1.mps.gz_['split', 'gready']__log.txt --cuts_output no --wcgn 1 --wcpl 30 --iters 1 --node_lim -1 --model_cuts off --trials_num 5 --submat_limit 512 --

KeyboardInterrupt: 

In [ ]:
# printing the duration

duration = dt.now() - starting_time
print(f"Work is finished!, duration is {duration}")